# Classic tier: BoW vs VLAD vs Fisher

Reads `results/runs.jsonl` — the committed record — and plots it. No pipeline logic
lives here; producing rows is `cbir evaluate`'s job.

To add points, from the repo root:

```bash
cbir evaluate --dataset roxford5k --sweep-k 16 64 256  vlad   --seed 0
cbir evaluate --dataset roxford5k --sweep-k 16 64      fisher --seed 0
cbir evaluate --dataset roxford5k --sweep-k 1000 20000 bow    --seed 0
```

Every run appends a row, so re-running this notebook picks them up. Runs that are
still in flight simply don't appear yet.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from cbir.eval.frame import tidy_rows
from cbir.eval.results import latest

# Categorical palette slots 1-3, used unchanged. Colour follows the *technique*, never
# its rank, so adding a technique or filtering one out never repaints the others.
COLOR = {"bow": "#2a78d6", "vlad": "#eb6834", "fisher": "#1baf7a"}
LABEL = {"bow": "BoW", "vlad": "VLAD", "fisher": "Fisher"}
ORDER = ["bow", "vlad", "fisher"]
PROTOCOLS = ["easy", "medium", "hard"]

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": MUTED, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10,
    "figure.dpi": 120,
})


def style(ax, title=None, xlabel=None, ylabel=None):
    """Recessive grid behind the marks, no top/right spines, mAP always from 0."""
    ax.set_axisbelow(True)
    ax.grid(True, axis="y", alpha=0.9)
    if title:
        ax.set_title(title, loc="left", fontsize=11, pad=8)
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    return ax

## The data

`latest()` collapses re-runs of a configuration to the most recent row, so a
re-measured point replaces rather than duplicates. This table is also the
**accessible view** of every figure below — nothing is encoded by colour alone.

In [ ]:
df = pd.DataFrame(tidy_rows(records=list(latest().values())))
df["technique"] = pd.Categorical(df["technique"], categories=ORDER, ordered=True)
df = df.sort_values(["technique", "k", "protocol"]).reset_index(drop=True)

print(f"{df['commit'].nunique()} commit(s), {len(df) // 3} run(s)")
df[["dataset", "technique", "k", "dim", "protocol", "map", "mp_at_k", "num_excluded", "seconds"]]

## mAP vs vocabulary size

One panel per protocol (small multiples) rather than one axis carrying all nine
lines. `k` is log-scaled because the swept values span orders of magnitude.

Read this chart knowing that **equal `k` is not equal capacity** — see the next
figure, which is the fairer axis.

In [ ]:
def line_panels(df, x, xlabel, metric="map"):
    subset = df.dropna(subset=[x, metric])
    present = [t for t in ORDER if not subset[subset["technique"] == t].empty]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)

    for ax, protocol in zip(axes, PROTOCOLS):
        rows = subset[subset["protocol"] == protocol]
        for technique in present:
            series = rows[rows["technique"] == technique].sort_values(x)
            if series.empty:
                continue
            ax.plot(series[x], series[metric], color=COLOR[technique], linewidth=2,
                    marker="o", markersize=5.5, label=LABEL[technique], zorder=3)
            # Direct label on the last point: three of the light-mode palette slots sit
            # under 3:1 on white, so identity must not rest on the swatch alone.
            last = series.iloc[-1]
            ax.annotate(LABEL[technique], (last[x], last[metric]), textcoords="offset points",
                        xytext=(6, 0), fontsize=8, color=MUTED, va="center")
        ax.set_xscale("log")
        style(ax, title=protocol.capitalize(), xlabel=xlabel)

    axes[0].set_ylabel("mAP" if metric == "map" else metric.replace("_", " "))
    axes[0].set_ylim(bottom=0)
    # A legend only once there are two or more series -- with one, the direct label
    # already names it and a legend box would just be a second copy sitting on the data.
    if len(present) > 1:
        axes[-1].legend(frameon=False, fontsize=9, loc="best")
    fig.tight_layout()
    return fig


if df["k"].notna().any():
    line_panels(df, "k", "vocabulary size k (log)")
else:
    print("no runs recorded yet — run the sweep above")

## mAP vs descriptor dimensionality

The comparison axis that is actually fair. BoW's vector is `k` long, VLAD's is
`k x 128`, Fisher's is `2 x k x 128` — so at k=64 they are 64-, 8192- and
16384-dimensional respectively. Plotting against `k` alone flatters BoW by two
orders of magnitude; this asks the real question, *what does each method get per
stored float?*

In [ ]:
if df["dim"].notna().any():
    line_panels(df, "dim", "descriptor dimensionality (log)")
else:
    print("no runs recorded yet — run the sweep above")

## Best configuration per technique

Each technique at whichever `k` maximises Medium mAP, across the three protocols.
The Easy > Medium > Hard ordering is a sanity check: any technique that breaks it
has a bug, not a result.

In [ ]:
medium = df[df["protocol"] == "medium"].dropna(subset=["map"])

if not medium.empty:
    best_k = medium.loc[medium.groupby("technique", observed=True)["map"].idxmax(), ["technique", "k"]]
    best = df.merge(best_k, on=["technique", "k"])

    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    techniques = [t for t in ORDER if t in set(best["technique"])]
    width = 0.8 / max(len(techniques), 1)

    for i, technique in enumerate(techniques):
        rows = best[best["technique"] == technique].set_index("protocol").reindex(PROTOCOLS)
        offset = (i - (len(techniques) - 1) / 2) * width
        positions = [p + offset for p in range(len(PROTOCOLS))]
        k = int(rows["k"].dropna().iloc[0])
        bars = ax.bar(positions, rows["map"], width=width * 0.92, color=COLOR[technique],
                      label=f"{LABEL[technique]} (k={k})", zorder=3)
        # Value labels rather than a number on every gridline; text stays in ink, not
        # the series colour.
        for bar, value in zip(bars, rows["map"]):
            if pd.notna(value):
                ax.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                            textcoords="offset points", xytext=(0, 3),
                            ha="center", fontsize=8, color=MUTED)

    ax.set_xticks(range(len(PROTOCOLS)), [p.capitalize() for p in PROTOCOLS])
    # Same rule as the line panels: with one technique the title carries its identity
    # and its k, so no legend box.
    if len(techniques) > 1:
        ax.legend(frameon=False, fontsize=9)
        title = "Best k per technique, roxford5k"
    else:
        only = best.iloc[0]
        title = f"{LABEL[only['technique']]} k={int(only['k'])}, roxford5k"
    style(ax, title=title, ylabel="mAP")
    ax.set_ylim(bottom=0)
    fig.tight_layout()
else:
    print("no runs recorded yet — run the sweep above")

## Caveats

- Every row here is **roxford5k**, with the vocabulary trained on rparis6k. The
  reverse direction needs its own runs and its own codebooks.
- `num_excluded` in the table is not cosmetic: Easy drops 2 of 70 queries (they have
  no easy positives), so its mAP is an average over 68. Protocols with different
  exclusion counts are not directly comparable.
- Fisher's GMM is fitted on a seeded subsample (`params.sample`), not the full
  held-out pool. Points at different `sample` values are different models and share
  no line.
- These are single-seed runs. Nothing here shows run-to-run variance, so treat small
  gaps between techniques as unresolved rather than real.

## Save the README figure

The README embeds one chart — mAP against descriptor dimensionality, the axis on
which the three techniques are actually comparable. It is regenerated here rather
than exported by hand, so it cannot drift from `results/runs.jsonl`.


In [ ]:
from pathlib import Path

DOCS = Path("..") / "docs"
DOCS.mkdir(exist_ok=True)

fig = line_panels(df, "dim", "descriptor dimensionality (log)")
# White rather than transparent: GitHub renders READMEs on a dark background too,
# and a transparent PNG would leave the dark ink invisible there.
fig.savefig(DOCS / "classic_comparison.png", dpi=110, bbox_inches="tight", facecolor="white")
print("wrote", (DOCS / "classic_comparison.png").resolve())
